# PCOS Survey EDA and External-Style Non-Invasive Pattern Review

## Introduction
This notebook is performing the exploratory analysis for the cleaned PCOS survey dataset. The survey table is providing a self-reported, non-clinical view of the syndrome and is therefore supporting the later external-style validation story of the MSc project.

The notebook is using `cleaned_data/PCOS_survey_cleaned.csv`. Because the dataset is self-reported, the analysis is being framed more cautiously than the clinical notebook. The goal is not proving clinical equivalence, but checking whether the main non-invasive pattern still appears in a noisier real-world style dataset.

## Research Positioning
This notebook is not merging the survey table row-wise with the clinical cohort. It is instead examining whether symptom prevalence, menstrual-pattern disruption, and BMI-related burden are still concentrating in survey participants who report PCOS.


## Reproducibility Setup and Path Configuration

This section is preparing the environment, the color palette, and the survey-specific figure-export directory.


In [ ]:
# Importing the libraries is supporting survey-table analysis, plotting, and notebook display.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Fixing the random seed is keeping reruns reproducible.
np.random.seed(42)

# Configuring the plotting theme is keeping the visuals consistent with the clinical and hormonal notebooks.
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", None)

# Resolving the project root is keeping the notebook portable across launch locations.
def resolve_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent, current.parent.parent]
    for candidate in candidates:
        if (candidate / "cleaned_data").exists() and (candidate / "images").exists():
            return candidate
    return current

PROJECT_ROOT = resolve_project_root()
DATA_PATH = PROJECT_ROOT / "cleaned_data" / "PCOS_survey_cleaned.csv"
IMAGE_DIR = PROJECT_ROOT / "images" / "eda" / "survey"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

PCOS_LABEL_ORDER = ["PCOS Negative", "PCOS Positive"]
PCOS_LABEL_PALETTE = {
    "PCOS Negative": "#3b82f6",
    "PCOS Positive": "#e76f51",
}

def save_figure(fig: plt.Figure, slug: str) -> Path:
    output_path = IMAGE_DIR / slug
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    return output_path

def grouped_numeric_summary(data: pd.DataFrame, feature: str) -> pd.DataFrame:
    return (
        data.groupby("pcos_label")[feature]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .rename(columns={"count": "n"})
        .round(3)
        .reset_index()
    )

def grouped_binary_prevalence(data: pd.DataFrame, feature: str) -> pd.DataFrame:
    summary = (
        data.groupby("pcos_label")[feature]
        .agg(["count", "sum", "mean"])
        .rename(columns={"count": "n", "sum": "positive_count", "mean": "prevalence"})
        .reset_index()
    )
    summary["prevalence_pct"] = (summary["prevalence"] * 100).round(1)
    return summary[["pcos_label", "n", "positive_count", "prevalence_pct"]]

def plot_numeric_by_target(
    data: pd.DataFrame,
    feature: str,
    ylabel: str,
    title: str,
    slug: str,
    kind: str = "box",
) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    if kind == "violin":
        sns.violinplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            palette=PCOS_LABEL_PALETTE,
            cut=0,
            inner=None,
            ax=ax,
        )
        sns.stripplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            color="#264653",
            alpha=0.30,
            size=3,
            jitter=0.20,
            ax=ax,
        )
    else:
        sns.boxplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            palette=PCOS_LABEL_PALETTE,
            ax=ax,
        )
        sns.stripplot(
            data=data,
            x="pcos_label",
            y=feature,
            order=PCOS_LABEL_ORDER,
            color="#264653",
            alpha=0.30,
            size=3,
            jitter=0.20,
            ax=ax,
        )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    save_figure(fig, slug)
    plt.show()

def plot_binary_prevalence(summary: pd.DataFrame, title: str, slug: str) -> None:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.barplot(
        data=summary,
        x="pcos_label",
        y="prevalence_pct",
        order=PCOS_LABEL_ORDER,
        palette=PCOS_LABEL_PALETTE,
        ax=ax,
    )
    for patch in ax.patches:
        height = patch.get_height()
        ax.annotate(
            f"{height:.1f}%",
            (patch.get_x() + patch.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=11,
            xytext=(0, 6),
            textcoords="offset points",
        )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Prevalence (%)")
    save_figure(fig, slug)
    plt.show()


## Data Loading and Derived Survey Features

This section is loading the cleaned survey table, checking the expected schema, and creating simple derived features that summarize non-invasive burden in the self-reported cohort.


In [ ]:
# Loading the cleaned survey dataset is bringing the self-reported PCOS table into memory for EDA.
df = pd.read_csv(DATA_PATH)

# Verifying the expected schema is protecting the notebook from upstream changes.
required_columns = [
    "pcos_y_n",
    "age_yrs",
    "weight_kg",
    "height_cm",
    "bmi",
    "cycle_regularity",
    "cycle_length_days",
    "months_between_periods",
    "period_duration_days",
    "weight_gain_y_n",
    "hair_growth_y_n",
    "skin_darkening_y_n",
    "hair_loss_y_n",
    "pimples_y_n",
    "fast_food_y_n",
    "regular_exercise_y_n",
    "mood_swings_y_n",
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

# Casting the survey fields to numeric is keeping the summary tables and plots explicit.
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

binary_columns = [
    "pcos_y_n",
    "cycle_regularity",
    "weight_gain_y_n",
    "hair_growth_y_n",
    "skin_darkening_y_n",
    "hair_loss_y_n",
    "pimples_y_n",
    "fast_food_y_n",
    "regular_exercise_y_n",
    "mood_swings_y_n",
]
for column in binary_columns:
    df[column] = df[column].astype(int)

# Creating readable labels and a simple burden score is supporting later survey-based interpretation.
df["pcos_label"] = df["pcos_y_n"].map({0: "PCOS Negative", 1: "PCOS Positive"})
df["low_exercise_flag"] = (1 - df["regular_exercise_y_n"]).astype(int)
df["irregular_cycle_flag"] = (1 - df["cycle_regularity"]).astype(int)
df["non_invasive_burden_score"] = (
    df["weight_gain_y_n"]
    + df["hair_growth_y_n"]
    + df["skin_darkening_y_n"]
    + df["hair_loss_y_n"]
    + df["pimples_y_n"]
    + df["fast_food_y_n"]
    + df["low_exercise_flag"]
    + df["irregular_cycle_flag"]
)

display(df.head())


## Question 1

### What is the cleaned survey dataset size, schema, and target balance?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the overview tables is showing the survey structure before the feature-level questions begin.
overview_q01 = pd.DataFrame(
    {
        "metric": ["row_count", "column_count", "missing_cells"],
        "value": [df.shape[0], df.shape[1], int(df.isna().sum().sum())],
    }
)
target_q01 = (
    df["pcos_label"]
    .value_counts()
    .reindex(PCOS_LABEL_ORDER)
    .rename_axis("pcos_label")
    .reset_index(name="count")
)
target_q01["percentage"] = (100 * target_q01["count"] / target_q01["count"].sum()).round(1)
schema_q01 = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values})
display(overview_q01)
display(schema_q01)
display(target_q01)


In [ ]:
# Plotting the survey target distribution is showing the class balance that later external-style validation will inherit.
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=target_q01,
    x="pcos_label",
    y="count",
    order=PCOS_LABEL_ORDER,
    palette=PCOS_LABEL_PALETTE,
    ax=ax,
)
ax.set_title("Question 1: Survey Target Distribution")
ax.set_xlabel("")
ax.set_ylabel("Participant Count")
save_figure(fig, "q01_survey_target_distribution.png")
plt.show()


### Insight


    The cleaned survey dataset is showing a smaller PCOS-positive group than the clinical cohort, which is expected in a self-reported setting. This balance is still adequate for descriptive EDA, but it is reinforcing the need to interpret later survey validation results with attention to class size and reporting noise.


## Question 2

### Does age differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying survey age by PCOS status before plotting.
summary_q02 = grouped_numeric_summary(df, "age_yrs")
display(summary_q02)


In [ ]:
# Plotting the survey age distribution is showing whether the target groups are materially age-shifted.
plot_numeric_by_target(
    data=df,
    feature="age_yrs",
    ylabel="Age (years)",
    title="Question 2: Survey Age by PCOS Status",
    slug="q02_age_vs_pcos_survey.png",
    kind="box",
)


### Insight

Age is showing only limited separation across the survey target groups, which is suggesting that age is more likely to behave as a contextual feature than as a dominant self-reported signal.


## Question 3

### Does BMI differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped BMI summary is quantifying the body-composition shift before plotting.
summary_q03 = grouped_numeric_summary(df, "bmi")
display(summary_q03)


In [ ]:
# Plotting the survey BMI distribution is showing whether adiposity signal survives in the self-reported cohort.
plot_numeric_by_target(
    data=df,
    feature="bmi",
    ylabel="Body Mass Index",
    title="Question 3: Survey BMI by PCOS Status",
    slug="q03_bmi_vs_pcos_survey.png",
    kind="box",
)


### Insight

BMI is showing a meaningful upward shift in the survey PCOS-positive group, which is directionally consistent with the clinical notebook. This matters because it suggests that adiposity-related signal is surviving even in a noisier self-reported dataset.


## Question 4

### Does cycle regularity prevalence differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the cycle-regularity prevalence table is quantifying how often regular cycles are being reported in each group.
summary_q04 = grouped_binary_prevalence(df, "cycle_regularity")
display(summary_q04)


In [ ]:
# Plotting the cycle-regularity prevalence is showing how strongly menstrual disruption is separating the survey groups.
plot_binary_prevalence(
    summary=summary_q04,
    title="Question 4: Regular Cycle Prevalence by Survey PCOS Status",
    slug="q04_cycle_regularity_prevalence.png",
)


### Insight

Regular-cycle prevalence is dropping sharply in the survey PCOS-positive group, which is providing one of the strongest self-reported signals in the dataset. This is clinically important because menstrual irregularity remains central to low-burden PCOS screening.


## Question 5

### Does cycle-length behavior differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `cycle_length_days` before plotting.
summary_q05 = grouped_numeric_summary(df, "cycle_length_days")
display(summary_q05)


In [ ]:
# Plotting the grouped distribution is showing how `cycle_length_days` is shifting across the survey target groups.
plot_numeric_by_target(
    data=df,
    feature="cycle_length_days",
    ylabel="Estimated Cycle Length (days)",
    title="Question 5: Does cycle-length behavior differ by survey PCOS status?",
    slug="q05_cycle_length_vs_pcos_survey.png",
    kind="box",
)


### Insight

The estimated cycle-length feature is showing a strong upward shift in the survey PCOS-positive group. That pattern is clinically important because longer or more disrupted cycles are central to PCOS screening and remain visible even in self-reported data.


## Question 6

### Do months between periods differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `months_between_periods` before plotting.
summary_q06 = grouped_numeric_summary(df, "months_between_periods")
display(summary_q06)


In [ ]:
# Plotting the grouped distribution is showing how `months_between_periods` is shifting across the survey target groups.
plot_numeric_by_target(
    data=df,
    feature="months_between_periods",
    ylabel="Months Between Periods",
    title="Question 6: Do months between periods differ by survey PCOS status?",
    slug="q06_months_between_periods_vs_pcos.png",
    kind="box",
)


### Insight

Months between periods is showing a clear upward shift among survey participants who report PCOS. This reinforces the same menstrual-disruption story seen in the clinical notebook, even though the survey variable is coarser and self-reported.


## Question 7

### Does period duration differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped summary is quantifying `period_duration_days` before plotting.
summary_q07 = grouped_numeric_summary(df, "period_duration_days")
display(summary_q07)


In [ ]:
# Plotting the grouped distribution is showing how `period_duration_days` is shifting across the survey target groups.
plot_numeric_by_target(
    data=df,
    feature="period_duration_days",
    ylabel="Period Duration (days)",
    title="Question 7: Does period duration differ by survey PCOS status?",
    slug="q07_period_duration_vs_pcos.png",
    kind="box",
)


### Insight

Period duration is showing a more modest shift than cycle spacing, which suggests that timing irregularity may be more informative than duration alone in this self-reported cohort.


## Question 8

### Does weight gain prevalence differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `weight_gain_y_n` is being reported in each survey target group.
summary_q08 = grouped_binary_prevalence(df, "weight_gain_y_n")
display(summary_q08)


In [ ]:
# Plotting the grouped prevalence is showing whether `weight_gain_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q08,
    title="Question 8: Weight Gain Prevalence by Survey PCOS Status",
    slug="q08_weight_gain_prevalence_survey.png",
)


### Insight

Weight gain is showing a clear prevalence jump in the survey PCOS-positive group, which is helping the self-reported dataset preserve an important metabolic and symptom burden signal.


## Question 9

### Does hair growth prevalence differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `hair_growth_y_n` is being reported in each survey target group.
summary_q09 = grouped_binary_prevalence(df, "hair_growth_y_n")
display(summary_q09)


In [ ]:
# Plotting the grouped prevalence is showing whether `hair_growth_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q09,
    title="Question 9: Hair Growth Prevalence by Survey PCOS Status",
    slug="q09_hair_growth_prevalence_survey.png",
)


### Insight

Hair growth is showing a strong prevalence shift in the survey cohort, which is supporting its role as an androgen-linked non-invasive marker even outside the clinical dataset.


## Question 10

### Does skin darkening prevalence differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `skin_darkening_y_n` is being reported in each survey target group.
summary_q10 = grouped_binary_prevalence(df, "skin_darkening_y_n")
display(summary_q10)


In [ ]:
# Plotting the grouped prevalence is showing whether `skin_darkening_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q10,
    title="Question 10: Skin Darkening Prevalence by Survey PCOS Status",
    slug="q10_skin_darkening_prevalence_survey.png",
)


### Insight

Skin darkening is again showing a notable upward prevalence in positive cases, which is directionally compatible with the clinical notebook and supportive of insulin-resistance-linked symptom burden.


## Question 11

### Does hair loss prevalence differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `hair_loss_y_n` is being reported in each survey target group.
summary_q11 = grouped_binary_prevalence(df, "hair_loss_y_n")
display(summary_q11)


In [ ]:
# Plotting the grouped prevalence is showing whether `hair_loss_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q11,
    title="Question 11: Hair Loss Prevalence by Survey PCOS Status",
    slug="q11_hair_loss_prevalence_survey.png",
)


### Insight

Hair loss is showing a moderate prevalence difference, suggesting that it remains useful but may not be as dominant as the stronger androgenic symptom features.


## Question 12

### Does pimples prevalence differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `pimples_y_n` is being reported in each survey target group.
summary_q12 = grouped_binary_prevalence(df, "pimples_y_n")
display(summary_q12)


In [ ]:
# Plotting the grouped prevalence is showing whether `pimples_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q12,
    title="Question 12: Pimples Prevalence by Survey PCOS Status",
    slug="q12_pimples_prevalence_survey.png",
)


### Insight

Pimples are showing a noticeable upward prevalence in the positive survey group. This matters because acne-like features are easy to self-report, although they may overlap with other androgen-linked symptoms.


## Question 13

### Does fast-food behavior differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `fast_food_y_n` is being reported in each survey target group.
summary_q13 = grouped_binary_prevalence(df, "fast_food_y_n")
display(summary_q13)


In [ ]:
# Plotting the grouped prevalence is showing whether `fast_food_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q13,
    title="Question 13: Fast Food Prevalence by Survey PCOS Status",
    slug="q13_fast_food_prevalence_survey.png",
)


### Insight

Fast-food behavior is showing only a modest difference in the survey cohort. This suggests it may contribute contextual lifestyle information but is unlikely to behave like a core screening feature on its own.


## Question 14

### Does regular exercise differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `regular_exercise_y_n` is being reported in each survey target group.
summary_q14 = grouped_binary_prevalence(df, "regular_exercise_y_n")
display(summary_q14)


In [ ]:
# Plotting the grouped prevalence is showing whether `regular_exercise_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q14,
    title="Question 14: Regular Exercise Prevalence by Survey PCOS Status",
    slug="q14_regular_exercise_prevalence_survey.png",
)


### Insight

Regular exercise is showing little separation across the survey groups, which is making it look more like a background behavior variable than a strong direct PCOS signal in this dataset.


## Question 15

### Does mood swings prevalence differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the grouped prevalence table is quantifying how often `mood_swings_y_n` is being reported in each survey target group.
summary_q15 = grouped_binary_prevalence(df, "mood_swings_y_n")
display(summary_q15)


In [ ]:
# Plotting the grouped prevalence is showing whether `mood_swings_y_n` is concentrating in the positive survey cohort.
plot_binary_prevalence(
    summary=summary_q15,
    title="Question 15: Mood Swings Prevalence by Survey PCOS Status",
    slug="q15_mood_swings_prevalence_survey.png",
)


### Insight

Mood swings are highly prevalent in both groups but are still showing a stronger concentration in the survey PCOS-positive cohort. This suggests potential relevance, although the high baseline prevalence means the feature may be sensitive but not especially specific.


## Question 16

### Does a simple non-invasive burden score differ by survey PCOS status?

This analysis is examining the cleaned dataset through a table-first and plot-second workflow.


In [ ]:
# Building the burden-score summary is quantifying cumulative survey-side symptom load before plotting.
summary_q16 = grouped_numeric_summary(df, "non_invasive_burden_score")
burden_distribution_q16 = (
    df.groupby(["pcos_label", "non_invasive_burden_score"])
    .size()
    .rename("count")
    .reset_index()
)
display(summary_q16)
display(burden_distribution_q16)


In [ ]:
# Plotting the burden score is showing whether multiple self-reported risk indicators are stacking together in positive cases.
plot_numeric_by_target(
    data=df,
    feature="non_invasive_burden_score",
    ylabel="Non-Invasive Burden Score",
    title="Question 16: Survey Non-Invasive Burden Score by PCOS Status",
    slug="q16_survey_burden_score.png",
    kind="violin",
)


### Insight

The burden score is showing whether multiple self-reported symptoms and behavior signals are stacking together inside the survey PCOS-positive group. A clear upward shift would support the idea that the non-invasive phenotype remains visible even when the data source is noisier.


## Generalization and Data-Quality Summary

The survey notebook is showing that the major non-invasive PCOS pattern is still visible in self-reported data, especially through BMI, menstrual disruption, weight gain, hair growth, skin darkening, and mood-related burden. These directions are broadly compatible with the stronger clinical signals, even though the survey dataset is noisier and more weakly structured.

Some features such as fast food and regular exercise are showing much weaker separation than the stronger reproductive and symptom variables. That matters because it suggests the later external-style validation phase should prioritize symptom burden and menstrual features over lifestyle variables that may be more confounded or inconsistently reported.

The survey table is therefore functioning as a realistic non-clinical stress test for the later non-invasive modeling workflow. It is not replacing the clinical dataset, but it is showing whether the same general pattern survives when measurement precision decreases.
